# 05 — Module E: Temporal Default Prediction
Does model performance degrade over time? Random vs temporal split.

In [ ]:
import sys, os

# Add project root to path
PROJECT_ROOT = "/Users/nando/PycharmProjects/PythonProject/SmartLender"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from src.data.loader import load_raw_data, split_data
from src.data.preprocessor import build_preprocessor, get_catboost_cat_indices
from src.data.features import add_temporal_features
from src.models.registry import get_classifiers
from src.models.trainer import run_arena
from src.evaluation.metrics import classification_metrics
from src.evaluation.comparison import save_comparison
from src.config import *

%matplotlib inline
mlflow.set_experiment('SmartLend')

## Load Data with Temporal Features

In [ ]:
df = load_raw_data(sample_size=200000)
df = add_temporal_features(df)

# Show temporal distribution
print(f"Date range: {df['issue_d'].min()} to {df['issue_d'].max()}")
print(f"\nYearly counts:")
print(df['issue_d'].dt.year.value_counts().sort_index())

## Temporal Split — Train pre-2017, Test 2017+

In [ ]:
X_train_t, X_test_t, y_train_t, y_test_t = split_data(df, TARGET_CLASSIFICATION, temporal=True)
print(f"Temporal split:")
print(f"  Train: {X_train_t.shape} (pre-2017)")
print(f"  Test:  {X_test_t.shape} (2017+)")
print(f"  Train default rate: {y_train_t.mean():.3f}")
print(f"  Test default rate:  {y_test_t.mean():.3f}")

## Run Arena with Temporal Split

In [ ]:
preprocessor_t = build_preprocessor(X_train_t)
cat_indices_t = get_catboost_cat_indices(X_train_t)
classifiers_t = get_classifiers()

temporal_df, temporal_results = run_arena(
    models=classifiers_t,
    preprocessor=preprocessor_t,
    X_train=X_train_t, y_train=y_train_t,
    X_test=X_test_t, y_test=y_test_t,
    metrics_fn=classification_metrics,
    cat_feature_indices=cat_indices_t,
    module_name='module_e',
)

temporal_df = temporal_df.sort_values('auc_roc', ascending=False)
print("\nTemporal Split Results:")
print(temporal_df.to_string(index=False))

## Random Split — Same Data (For Comparison)

In [ ]:
X_train_r, X_test_r, y_train_r, y_test_r = split_data(df, TARGET_CLASSIFICATION, temporal=False)
preprocessor_r = build_preprocessor(X_train_r)
cat_indices_r = get_catboost_cat_indices(X_train_r)
classifiers_r = get_classifiers()

random_df, random_results = run_arena(
    models=classifiers_r,
    preprocessor=preprocessor_r,
    X_train=X_train_r, y_train=y_train_r,
    X_test=X_test_r, y_test=y_test_r,
    metrics_fn=classification_metrics,
    cat_feature_indices=cat_indices_r,
    module_name='module_e_random',
)

random_df = random_df.sort_values('auc_roc', ascending=False)
print("\nRandom Split Results:")
print(random_df.to_string(index=False))

## Temporal vs Random — Metric Degradation

In [ ]:
# Merge by algorithm
merged = temporal_df[['algorithm', 'auc_roc', 'f1']].merge(
    random_df[['algorithm', 'auc_roc', 'f1']],
    on='algorithm',
    suffixes=('_temporal', '_random')
)
merged['auc_drop'] = merged['auc_roc_random'] - merged['auc_roc_temporal']
merged['f1_drop'] = merged['f1_random'] - merged['f1_temporal']
merged = merged.sort_values('auc_drop', ascending=False)

print("\nMetric Degradation (Random - Temporal):")
print(merged.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(merged))
width = 0.35
ax.bar([i - width/2 for i in x], merged['auc_roc_random'], width, label='Random Split', color='#2ecc71')
ax.bar([i + width/2 for i in x], merged['auc_roc_temporal'], width, label='Temporal Split', color='#e74c3c')
ax.set_xticks(x)
ax.set_xticklabels(merged['algorithm'], rotation=45, ha='right')
ax.set_ylabel('AUC-ROC')
ax.set_title('Random vs Temporal Split — AUC-ROC Comparison')
ax.legend()
plt.tight_layout()
plt.savefig('results/figures/module_e_temporal_vs_random.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n⚠️ Random split inflates metrics because future data leaks into training.")
print("   Temporal split is the honest evaluation for production deployment.")

## Save Results

In [ ]:
save_comparison(temporal_df, 'module_e')